# 04 — Panel Scoring (Pre / Post Windows)

Scores each panel user's text in the **pre-baseline** (August) and **post-outcome** (December–May)
windows using the trained SVM classifiers, then merges with exposure labels to build the analysis panel.

**Inputs:**
- `data/processed_v2/posts_clean.jsonl` + `comments_clean.jsonl` (from notebook 01)
- `data/processed_v2/exposure_labels_v2.parquet` (from notebook 03)
- `models/clf_anxiety.joblib`, `clf_depression.joblib`, `clf_stress.joblib` (from notebook 02)

**Output:** `data/processed_v2/panel_scores_v2.parquet`

| Column | Description |
|--------|-------------|
| author | Reddit username |
| cycle | 1 or 2 |
| exposed | bool — whether user commented on an anchor thread |
| pre_mh_score | Mean SVM MH score across Aug posts/comments |
| pre_n_posts | Count of posts/comments in Aug |
| post_mh_score | Mean SVM MH score across Dec–May posts/comments |
| post_n_posts | Count of posts/comments in Dec–May |

**Cycle windows:**

| | Cycle 1 | Cycle 2 |
|-|---------|---------|
| Pre baseline | Aug 1–31, 2023 | Aug 1–31, 2024 |
| Post outcome | Dec 1, 2023 – May 31, 2024 | Dec 1, 2024 – May 31, 2025 |

In [13]:
import json
import numpy as np
import pandas as pd
import joblib
from datetime import datetime, timezone
from pathlib import Path

ROOT      = Path('..').resolve()
DATA_V2   = ROOT / 'data' / 'processed_v2'
MODEL_DIR = ROOT / 'models'

POSTS_CLEAN    = DATA_V2 / 'posts_clean.jsonl'
COMMENTS_CLEAN = DATA_V2 / 'comments_clean.jsonl'
EXPOSURE_PATH  = DATA_V2 / 'exposure_labels_v2.parquet'
OUT_PATH       = DATA_V2 / 'panel_scores_v2.parquet'

# Cycle windows (inclusive on both ends)
CYCLES = {
    1: {
        'pre_start':  datetime(2023,  8,  1, tzinfo=timezone.utc),
        'pre_end':    datetime(2023,  8, 31, 23, 59, 59, tzinfo=timezone.utc),
        'post_start': datetime(2023, 12,  1, tzinfo=timezone.utc),
        'post_end':   datetime(2024,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
    2: {
        'pre_start':  datetime(2024,  8,  1, tzinfo=timezone.utc),
        'pre_end':    datetime(2024,  8, 31, 23, 59, 59, tzinfo=timezone.utc),
        'post_start': datetime(2024, 12,  1, tzinfo=timezone.utc),
        'post_end':   datetime(2025,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
}

print('Paths OK:', all(p.exists() for p in [POSTS_CLEAN, COMMENTS_CLEAN, EXPOSURE_PATH]))

Paths OK: True


## 1) Load panel users

In [14]:
exposure = pd.read_parquet(EXPOSURE_PATH)
print(f'Panel users: {exposure["author"].nunique():,}  |  rows: {len(exposure):,}')
print(exposure['exposed'].value_counts())
panel_users = set(exposure['author'])

Panel users: 20,932  |  rows: 21,730
exposed
False    19697
True      2033
Name: count, dtype: int64


## 2) Load clean corpus and assign cycle windows

In [15]:
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

posts    = load_jsonl(POSTS_CLEAN)
comments = load_jsonl(COMMENTS_CLEAN)
print(f'Posts: {len(posts):,}  |  Comments: {len(comments):,}')

Posts: 78,961  |  Comments: 467,986


In [16]:
# Combine into a single list; keep only panel users and relevant windows
def parse_dt(s):
    dt = datetime.fromisoformat(s)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt


def assign_window(dt):
    """Return (cycle, window) tuple or None if outside all windows."""
    for cycle, w in CYCLES.items():
        if w['pre_start'] <= dt <= w['pre_end']:
            return cycle, 'pre'
        if w['post_start'] <= dt <= w['post_end']:
            return cycle, 'post'
    return None


records = []
for r in posts + comments:
    author = r.get('author')
    if author not in panel_users:
        continue
    try:
        dt = parse_dt(r['created_dt'])
    except Exception:
        continue
    result = assign_window(dt)
    if result is None:
        continue
    cycle, window = result
    records.append({'author': author, 'cycle': cycle, 'window': window,
                    'created_dt': r['created_dt'],
                    'clean_text': r.get('clean_text', '')})

corpus = pd.DataFrame(records)
print(f'Panel records in scoring windows: {len(corpus):,}')
print(corpus.groupby(['cycle', 'window']).size())

Panel records in scoring windows: 147,569
cycle  window
1      post      72623
       pre        5794
2      post      64020
       pre        5132
dtype: int64


## 3) Load SVM classifiers

In [17]:
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Classifiers loaded.')

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def score_texts_all(texts):
    """Return (anx, dep, str, mean) arrays for a list of strings."""
    anx  = sigmoid(clf_anx.decision_function(texts))
    dep  = sigmoid(clf_dep.decision_function(texts))
    str_ = sigmoid(clf_str.decision_function(texts))
    mean = np.stack([anx, dep, str_], axis=1).mean(axis=1)
    return anx, dep, str_, mean

Classifiers loaded.


## 4) Score all records

In [18]:
texts = corpus['clean_text'].tolist()
print(f'Scoring {len(texts):,} records...')
corpus['anx_score'], corpus['dep_score'], corpus['str_score'], corpus['mean_mh_score'] = score_texts_all(texts)
print('Done.')
corpus[['anx_score', 'dep_score', 'str_score', 'mean_mh_score']].describe().round(4)

Scoring 147,569 records...
Done.


,anx_score,dep_score,str_score,mean_mh_score
count,147569.0000,147569.0000,147569.0000,147569.0000
mean,0.4242,0.4201,0.4580,0.4341
std,0.0808,0.0881,0.0815,0.0771
min,0.0652,0.0563,0.0561,0.0700
25%,0.3705,0.3611,0.4037,0.3832
50%,0.4221,0.4199,0.4583,0.4341
75%,0.4741,0.4756,0.5093,0.4823
max,0.9881,0.8664,0.9398,0.8302


## 4b) Save post-level scores (for post-level DiD in NB06)

In [19]:
# Save individual post/comment records with scores before aggregation.
# Used by NB06 for post-level DiD (recovers ~147K observations vs 1,094 user-means).
POST_LEVEL_PATH = DATA_V2 / 'post_level_scores_v2.parquet'

post_level = corpus[['author', 'cycle', 'window', 'created_dt',
                      'anx_score', 'dep_score', 'str_score', 'mean_mh_score']].copy()
post_level['created_dt'] = pd.to_datetime(post_level['created_dt'], utc=True)

post_level.to_parquet(POST_LEVEL_PATH, index=False)
print(f'Saved {len(post_level):,} post-level rows → {POST_LEVEL_PATH}')
print(post_level.groupby(['cycle', 'window']).size())

Saved 147,569 post-level rows → /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/post_level_scores_v2.parquet
cycle  window
1      post      72623
       pre        5794
2      post      64020
       pre        5132
dtype: int64


## 4c) Compute dose-response data (# anchor-thread comments per user)

In [20]:
import json as _json

ANCHOR_PATH   = DATA_V2 / 'anchor_posts_v2.parquet'
DOSE_PATH     = DATA_V2 / 'dose_exposure_v2.parquet'
COMMENTS_PATH = ROOT / 'data' / 'processed_v2' / 'comments_clean.jsonl'

anchor_posts_df = pd.read_parquet(ANCHOR_PATH, columns=['id', 'cycle'])
anchor_ids      = set(anchor_posts_df['id'].astype(str))

# Load only comments that are on anchor threads
dose_rows = []
with open(COMMENTS_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = _json.loads(line)
        if r.get('post_id') in anchor_ids and r.get('author') in panel_users:
            dose_rows.append({'author': r['author'], 'post_id': r['post_id']})

dose_comments = pd.DataFrame(dose_rows)

# Map post_id → cycle
pid_to_cycle = anchor_posts_df.set_index('id')['cycle'].to_dict()
dose_comments['cycle'] = dose_comments['post_id'].map(pid_to_cycle)

# Count comments per (author, cycle)
dose = (
    dose_comments.groupby(['author', 'cycle'])
    .size()
    .reset_index(name='n_anchor_comments')
)
dose['log1p_n_anchor'] = np.log1p(dose['n_anchor_comments'])

dose.to_parquet(DOSE_PATH, index=False)
print(f'Saved {len(dose):,} dose records → {DOSE_PATH}')
print(dose['n_anchor_comments'].describe().round(2))

Saved 2,300 dose records → /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/dose_exposure_v2.parquet
count    2300.00
mean        2.02
std         2.81
min         1.00
25%         1.00
50%         1.00
75%         2.00
max        49.00
Name: n_anchor_comments, dtype: float64


## 5) Aggregate per (author, cycle, window)

In [21]:
agg = (
    corpus
    .groupby(['author', 'cycle', 'window'])
    .agg(
        mh_score  = ('mean_mh_score', 'mean'),
        anx_score = ('anx_score',     'mean'),
        dep_score = ('dep_score',     'mean'),
        str_score = ('str_score',     'mean'),
        n_posts   = ('mean_mh_score', 'count'),
    )
    .reset_index()
)

def pivot_window(window_label, prefix):
    sub = agg[agg['window'] == window_label].drop(columns='window')
    return sub.rename(columns={
        'mh_score':  f'{prefix}_mh_score',
        'anx_score': f'{prefix}_anx_score',
        'dep_score': f'{prefix}_dep_score',
        'str_score': f'{prefix}_str_score',
        'n_posts':   f'{prefix}_n_posts',
    })

pre  = pivot_window('pre',  'pre')
post = pivot_window('post', 'post')

scores = pre.merge(post, on=['author', 'cycle'], how='inner')
print(f'Users with both pre and post observations: {len(scores):,}')

Users with both pre and post observations: 1,508


## 6) Merge with exposure labels

In [22]:
panel = exposure.merge(scores, on=['author', 'cycle'], how='inner')
print(f'Final panel rows: {len(panel):,}')
print(f'Unique users:     {panel["author"].nunique():,}')
print(f'\nCoverage: {100 * panel["author"].nunique() / len(panel_users):.1f}% of panel users have pre+post scores')
print('\nExposure breakdown:')
print(panel.groupby(['cycle', 'exposed']).size())

Final panel rows: 1,423
Unique users:     1,368

Coverage: 6.5% of panel users have pre+post scores

Exposure breakdown:
cycle  exposed
1      False      629
       True       156
2      False      472
       True       166
dtype: int64


In [23]:
# Score distribution check
print('Pre-period MH scores:')
print(panel.groupby('exposed')['pre_mh_score'].describe().round(4))
print('\nPost-period MH scores:')
print(panel.groupby('exposed')['post_mh_score'].describe().round(4))

Pre-period MH scores:
          count    mean     std     min     25%     50%     75%     max
exposed                                                                
False    1101.0  0.4000  0.0658  0.1028  0.3596  0.4022  0.4393  0.6425
True      322.0  0.4073  0.0566  0.2416  0.3738  0.4062  0.4395  0.6612

Post-period MH scores:
          count    mean     std     min     25%     50%     75%     max
exposed                                                                
False    1101.0  0.4271  0.0510  0.1538  0.3985  0.4295  0.4561  0.6423
True      322.0  0.4333  0.0419  0.2312  0.4127  0.4318  0.4529  0.6550


## 7) Save

> **Approval gate:** Review coverage stats and score distributions above before running this cell.

In [24]:
out_cols = [
    'author', 'cycle', 'exposed',
    'pre_mh_score',  'pre_anx_score',  'pre_dep_score',  'pre_str_score',  'pre_n_posts',
    'post_mh_score', 'post_anx_score', 'post_dep_score', 'post_str_score', 'post_n_posts',
]
panel[out_cols].to_parquet(OUT_PATH, index=False)
print(f'Saved {len(panel):,} rows → {OUT_PATH}')
print('Columns:', out_cols)

Saved 1,423 rows → /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/panel_scores_v2.parquet
Columns: ['author', 'cycle', 'exposed', 'pre_mh_score', 'pre_anx_score', 'pre_dep_score', 'pre_str_score', 'pre_n_posts', 'post_mh_score', 'post_anx_score', 'post_dep_score', 'post_str_score', 'post_n_posts']
